# **CFPB Complaint Investigation Predictor**
## Notebook 2: Feature Engineering

### Objective
Transform the raw CFPB complaints dataset into a clean, 
model-ready feature set. This notebook builds on the findings 
from Notebook 1 to create meaningful features that capture 
the signals most likely to predict whether a complaint 
will be investigated.

### Key Tasks
- Consolidate inconsistent product category names
- Handle missing values strategically
- Create binary flags from sparse columns
- Encode categorical variables for modeling
- Define and validate the target variable
- Export clean dataset for model training

### Design Decisions from Notebook 1
- Credit reporting product names consolidated into one category
- `Consumer disputed?` excluded — discontinued after 2017
- `Tags` converted to binary flag `has_tag` — 95% missing
- `Consumer complaint narrative` flagged as `has_narrative` — 74% missing
- Class imbalance noted: 95.52% vs 4.48% — handled in Notebook 3

### Tools
- **DuckDB** — efficient SQL transformations on 8GB file
- **Pandas** — dataframe manipulation
- **Scikit-learn** — encoding and preprocessing

## **1. Setup & Connect**

In [1]:
# Import libraries 
import duckdb
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Create connection
conn = duckdb.connect()
file_path = '/home/ubuntu/data/complaints.csv'

# Create view
conn.execute(f"""
            CREATE OR REPLACE VIEW complaints AS
            Select * 
            From read_csv_auto('{file_path}', parallel=false)
""")
print('DuckDb connected and View created')

DuckDb connected and View created


## **2. Consolidate Product Names**
Credit reporting appeared under 3 different names due to CFPB renaming.
We consolidate into one category before feature engineering.

In [2]:
# Do transformation entrirely in DuckDb - no pandas loading
conn.execute("""
        CREATE OR REPLACE VIEW complaints_clean AS 
        Select 
            "Complaint ID",
            CASE
                WHEN "Product" ILIKE '%credit report%' THEN 'Credit Reporting'
                ELSE "Product"
            END AS product_clean,
            "Issue",
            "Sub-Issue",
            "Company",
            "State",
            "Tags",
            "Timely response?",
            "Company response to consumer",
            "Consumer complaint narrative",
            "Date received"
         From complaints               
""")

# Only oull aggregated result to verify - not full data
conn.execute("""
    Select 
        product_clean,
        Count(*) as count
    From complaints_clean
    Group By product_clean
    Order By count Desc
    Limit 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_clean,count
0,Credit Reporting,11607043
1,Debt collection,1063624
2,Mortgage,445448
3,Checking or savings account,357446
4,Credit card,303018
5,Credit card or prepaid card,206364
6,"Money transfer, virtual currency, or money ser...",174691
7,Student loan,124888
8,Vehicle loan or lease,92459
9,Bank account or service,86200


### Result: Product Consolidation
Credit reporting successfully consolidated from 3 names into one category.
`Credit Reporting` is now clearly the dominant product with 1.16M complaints — 
roughly 3x larger than the next category (Debt collection at 1.06M).

## **3. Create Binary Flags**

### 3.1 Has Tag
`Tags` column is 95% missing — only applies to federally protected groups 
(Servicemembers, Older Americans). We convert to a simple binary flag.

In [3]:
# create 'has tag' column on existing coplaints_clean view
conn.execute("""
        CREATE VIEW complaints_v2 AS
        Select *,
            CASE 
                WHEN "Tags" IS NULL THEN 0
                ELSE 1
            END as has_tag
        From complaints_clean
""")

# verify
conn.execute("""
    Select 
        has_tag,
        Count(*) as count
    From complaints_v2
    Group by has_tag
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,has_tag,count
0,0,13855593
1,1,737866


### Result: has_tag
Binary flag created successfully.
95% of complaints have no tag (0), 5% belong to federally protected groups (1).
This sparse but meaningful feature will be used directly in the model.

## 3.2 Has Narrative
`Consumer complaint narrative` is 74% missing — consumers optionally provide 
free text describing their complaint. Rather than attempting to impute this, 
we create a binary flag. The actual text will be used for LLM enrichment in Notebook 4.


In [4]:
# create has_narrative binary column form consumer complaint narrative 
conn.execute("""
        CREATE VIEW complaints_v3 AS
        Select *,
            CASE 
                WHEN "Consumer complaint narrative" IS NULL THEN 0
                ELSE 1
            END AS has_narrative
        From complaints_v2
""")
# verify 
conn.execute("""
        Select
            has_narrative,
            Count(*) as count
        From complaints_v3
        Group By has_narrative
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,has_narrative,count
0,0,10831654
1,1,3761805


### Result: has_narrative
74% of complaints have no narrative (0), 26% include a consumer written narrative (1).
The 376K complaints with narratives will be used for LLM enrichment in Notebook 4.

### 3.3 Timely Response
`Timely response?` indicates whether the company responded within CFPB's 
mandatory response window. Currently stored as True/False — we convert to 
1/0 integer for modeling compatibility. 
Untimely responses (0) are a strong signal for investigation.

In [5]:
# create timely_response binary
conn.execute("""
        CREATE VIEW complaints_v4 AS
        Select *,
            CASE
                WHEN "Timely response?"= TRUE THEN 1
                ELSE 0
            END AS timely_response
         From complaints_v3  
""")
# verify
conn.execute("""
        Select
            timely_response,
            Count(*) as count
        From complaints_v4
        Group by timely_response
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,timely_response,count
0,0,94616
1,1,14498843


### Result: timely_response
94,616 complaints (1%) received untimely responses — 
a strong signal for investigation despite being rare.

## 3.4. Target Variable Definition

We define `investigated = 1` based on signals identified during exploration:
- `Untimely response` — company missed CFPB mandatory response deadline
- `Closed with monetary relief` — serious enough for financial compensation
- `In progress` — still under active review

All other responses = `investigated = 0` (routine closures)

In [19]:
conn.execute("""
    CREATE VIEW complaints_v5 AS
    Select *,
        CASE
            WHEN "Company response to consumer" IN (
                'Untimely response',
                'Closed with monetary relief',
                'In progress'
            ) THEN 1
            ELSE 0
        END AS investigated
    From complaints_v4
""")
# Verify
conn.execute("""
    Select 
        investigated,
        Count(*) as count
    From complaints_v5
    Group by investigated
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,investigated,count
0,0,13939404
1,1,654055


## 4. Final Clean View — complaints_final

We consolidate all engineered features into one clean final view,
dropping raw columns that have been replaced by engineered versions:

- `Tags` → replaced by `has_tag` (binary flag)
- `Timely response?` → replaced by `timely_response` (0/1 integer)
- `Company response to consumer` → replaced by `investigated` (target variable)
- `Consumer complaint narrative` → replaced by `has_narrative` (binary flag)
  - Raw text preserved separately for LLM enrichment in Notebook 4

Final columns kept:
- **Identifiers:** `Complaint ID`, `Date received`
- **Categorical:** `product_clean`, `Issue`, `Sub-issue`, `Company`, `State`
- **Binary flags:** `has_tag`, `has_narrative`, `timely_response`
- **Target:** `investigated`

In [22]:
conn.execute("""
    CREATE VIEW complaints_final AS
    Select 
        "Complaint ID",
        product_clean,
        "Issue",
        "Sub-Issue",
        "Company",
        "State",
        "Date received",
        has_tag,
        has_narrative,
        timely_response,
        investigated
    From complaints_v5
""")
# Verify
conn.execute("""
    Select * 
    From complaints_final
    Limit 2
""").df()

,Complaint ID,product_clean,Issue,Sub-issue,Company,State,Date received,has_tag,has_narrative,timely_response,investigated
0,3730948,Credit Reporting,Incorrect information on your report,Information belongs to someone else,Experian Information Solutions Inc.,FL,2020-07-06,0,0,1,0
1,3477549,Credit card or prepaid card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,CAPITAL ONE FINANCIAL CORPORATION,CA,2019-12-26,0,0,1,0


In [23]:
conn.execute("""
    Select *
    From complaints_final
    Limit 1
""").df().columns.tolist()

['Complaint ID',
 'product_clean',
 'Issue',
 'Sub-issue',
 'Company',
 'State',
 'Date received',
 'has_tag',
 'has_narrative',
 'timely_response',
 'investigated']

## 5. Final Verification

Before exporting, we verify that `complaints_final` matches 
our Notebook 1 findings:

- Total rows should be ~1.4M
- Class balance should be ~95.52% not investigated (0) vs 4.48% investigated (1)

Any discrepancy here indicates an issue in the view chain that must be fixed
before moving to modeling.

In [24]:
# Row check
conn.execute("""
    Select Count(*) as total
    From complaints_final
    """).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total
0,14593459


In [26]:
# Class balance
conn.execute("""
    Select 
        investigated,
        COUNT(*) as count,
        ROUND(100.00 * COUNT(*) / SUM(COUNT(*))OVER(),2) as percentage
    From complaints_final
    Group by investigated
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,investigated,count,percentage
0,0,13939404,95.52
1,1,654055,4.48


## 6. Export Final Dataset
Export `complaints_final` to parquet format for use in Notebook 3 modeling.
Parquet is preferred over CSV for speed, compression, and data type preservation.

In [27]:
conn.execute("""
    COPY complaints_final 
    TO '/home/ubuntu/data/complaints_final.parquet' 
    (FORMAT PARQUET)
""")

print("Export complete!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Export complete!


In [29]:
# Verify the file export
import os

size = os.path.getsize('/home/ubuntu/data/complaints_final.parquet')
print(f"File size: {size / (1024**2):.2f} MB")

File size: 167.22 MB


### Result: Export Complete
Original CSV: 8GB → Parquet: 167MB (98% size reduction)
Parquet preserves all data types and loads significantly 
faster for model training in Notebook 3.

## 7. Notebook 2 Summary

### Completed
- ✅ Consolidated credit reporting product names (3 → 1 category)
- ✅ Created binary flags: `has_tag`, `has_narrative`, `timely_response`
- ✅ Defined target variable `investigated` (1=investigated, 0=routine)
- ✅ Dropped raw columns replaced by engineered features
- ✅ Exported clean dataset (167MB parquet)

### Missing Values Status
- ✅ `has_tag`, `has_narrative`, `timely_response` — handled via binary flags
- ⚠️ Categorical columns (`Issue`, `Sub-issue`, `Company`, `State`) — 
  NULL values to be handled during encoding in Notebook 3
  
### Next — Notebook 3: Model Training
- Load parquet file
- Encode categorical variables
- Train baseline logistic regression
- Train XGBoost
- Evaluate with precision, recall, F1
- SHAP explainability